# Sponsor/cosponsor network graphs

Explore [`congressgov.services.export.graph`](../../src/congressgov/services/export/graph/) — raw sponsorship events, filtered graph projections, drill-down evidence, and Sigma.js export.

Install optional dependencies:

```bash
pip install "congressgov[export,graph,viz]"
# from source: pip install -e ".[export,graph,viz]"
```

The notebook runs **offline sample data** first (no API key). Later cells optionally fetch live bills when `CONGRESS_API_KEY` is set.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from IPython.display import JSON, display

# Re-run this cell after updating the SDK so new graph exports are picked up.
import sys

for _module_name in list(sys.modules):
    if _module_name.startswith("congressgov.services.export.graph"):
        del sys.modules[_module_name]

from congressgov.models.entities.bill import Bill
from congressgov.services.export.graph import (
    EgoNetworkConfig,
    GraphChamberFilter,
    GraphExploreConfig,
    GraphProjectionType,
    GraphRenderConfig,
    GraphVisualizer,
    LayoutAlgorithm,
    NodeColorMode,
    build_ego_graph,
    build_graph_slice,
    build_matrix,
    build_member_summary,
    build_projection_key,
    create_member_label_resolver,
    export_graph_slice_csv,
    export_graph_slice_graphml,
    export_graph_slice_json,
    export_interactive_html,
    get_edge_evidence,
    graph_slice_to_api_payload,
    graph_slice_to_sigma_payload,
    rank_relationships,
    sponsorship_events_from_bill,
    sponsorship_events_from_bills,
)

OUTPUT_DIR = Path("output/network_graph")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_CONFIG = GraphExploreConfig(congress=118, min_weight=1, max_edges=50)
print("Projection key:", build_projection_key(BASE_CONFIG))

## Sample bills → raw events

Each `SponsorshipEvent` is one sponsor/cosponsor pair on one bill. This raw layer stays intact so edges can be traced back to bill evidence.

In [ ]:
def sample_bills() -> list[Bill]:
    return [
        Bill.model_validate(
            {
                "congress": 118,
                "number": 1,
                "type": "HR",
                "title": "Sample Health Bill",
                "originChamber": "House",
                "introducedDate": "2023-01-09",
                "policyArea": {"name": "Health"},
                "sponsors": [
                    {
                        "bioguideId": "S001",
                        "fullName": "Rep. Sponsor One",
                        "party": "D",
                        "state": "NJ",
                        "district": 7,
                        "url": "https://api.congress.gov/v3/member/S001",
                    }
                ],
                "cosponsors": [
                    {
                        "bioguideId": "C001",
                        "fullName": "Rep. Cosponsor One",
                        "party": "D",
                        "state": "NJ",
                        "isOriginalCosponsor": True,
                        "sponsorshipDate": "2023-01-10",
                        "url": "https://api.congress.gov/v3/member/C001",
                    },
                    {
                        "bioguideId": "C002",
                        "fullName": "Rep. Cosponsor Two",
                        "party": "R",
                        "state": "TX",
                        "isOriginalCosponsor": False,
                        "sponsorshipDate": "2023-02-01",
                        "url": "https://api.congress.gov/v3/member/C002",
                    },
                ],
            }
        ),
        Bill.model_validate(
            {
                "congress": 118,
                "number": 2,
                "type": "HR",
                "title": "Second Health Bill",
                "originChamber": "House",
                "policyArea": {"name": "Health"},
                "sponsors": [
                    {
                        "bioguideId": "S001",
                        "fullName": "Rep. Sponsor One",
                        "party": "D",
                        "state": "NJ",
                        "url": "https://api.congress.gov/v3/member/S001",
                    }
                ],
                "cosponsors": [
                    {
                        "bioguideId": "C001",
                        "fullName": "Rep. Cosponsor One",
                        "party": "D",
                        "state": "NJ",
                        "url": "https://api.congress.gov/v3/member/C001",
                    },
                    {
                        "bioguideId": "C003",
                        "fullName": "Rep. Cosponsor Three",
                        "party": "D",
                        "state": "CA",
                        "url": "https://api.congress.gov/v3/member/C003",
                    },
                ],
            }
        ),
        Bill.model_validate(
            {
                "congress": 118,
                "number": 10,
                "type": "S",
                "title": "Senate Sample Bill",
                "originChamber": "Senate",
                "policyArea": {"name": "Transportation"},
                "sponsors": [
                    {
                        "bioguideId": "S010",
                        "fullName": "Sen. Ten",
                        "party": "D",
                        "state": "DE",
                        "url": "https://api.congress.gov/v3/member/S010",
                    }
                ],
                "cosponsors": [
                    {
                        "bioguideId": "S011",
                        "fullName": "Sen. Eleven",
                        "party": "D",
                        "state": "RI",
                        "url": "https://api.congress.gov/v3/member/S011",
                    }
                ],
            }
        ),
    ]


events = sponsorship_events_from_bills(sample_bills())
events_df = pd.DataFrame([event.model_dump(mode="json") for event in events])
display(events_df[[
    "bill_id",
    "sponsor_bioguide_id",
    "cosponsor_bioguide_id",
    "policy_area",
    "is_original_cosponsor",
    "is_active",
]])

## Build a graph slice

The default projection aggregates directed **sponsor → cosponsor** edges with weights for active, original, late, withdrawn, and cross-party relationships.

In [ ]:
graph = build_graph_slice(events, BASE_CONFIG)

print(f"Nodes: {len(graph.nodes)}")
print(f"Edges shown: {graph.limits.shown_edges} / {graph.limits.edge_count_before_limit}")
print(f"Truncated: {graph.limits.was_truncated}")

edges_df = pd.DataFrame([edge.model_dump(mode="json") for edge in graph.edges])
display(edges_df[["source", "target", "weight", "original_weight", "cross_party_weight", "bill_count"]])

## Visualize the projection

Node size reflects weighted degree. Edge width reflects relationship weight. Party colors are a simple heuristic for exploration only.

In [ ]:
PARTY_COLORS = {"D": "#2563eb", "R": "#dc2626", "I": "#7c3aed"}


def draw_graph_slice(graph_slice, *, title: str = "Sponsor → cosponsor network") -> None:
    digraph = nx.DiGraph()
    for node in graph_slice.nodes:
        digraph.add_node(
            node.id,
            label=node.label or node.id,
            party=node.party,
            size=max(node.size, 1.0),
        )
    for edge in graph_slice.edges:
        digraph.add_edge(edge.source, edge.target, weight=edge.weight)

    pos = {
        node.id: (node.x or 0.0, node.y or 0.0)
        for node in graph_slice.nodes
    }
    if not pos:
        pos = nx.spring_layout(digraph, seed=42)

    node_colors = [PARTY_COLORS.get(digraph.nodes[node_id].get("party"), "#64748b") for node_id in digraph.nodes]
    node_sizes = [300 + 120 * digraph.nodes[node_id].get("size", 1.0) for node_id in digraph.nodes]
    edge_widths = [0.5 + 0.8 * data.get("weight", 1) for _, _, data in digraph.edges(data=True)]

    fig, ax = plt.subplots(figsize=(9, 7))
    nx.draw_networkx_edges(digraph, pos, width=edge_widths, alpha=0.45, arrows=True, ax=ax)
    nx.draw_networkx_nodes(digraph, pos, node_color=node_colors, node_size=node_sizes, ax=ax)
    labels = {node_id: digraph.nodes[node_id].get("label", node_id) for node_id in digraph.nodes}
    nx.draw_networkx_labels(digraph, pos, labels=labels, font_size=8, ax=ax)
    ax.set_title(title)
    ax.axis("off")
    plt.show()


draw_graph_slice(graph)

## Filter the graph

Try raising `min_weight`, filtering by chamber or policy area, or keeping only cross-party ties.

In [ ]:
filtered = build_graph_slice(
    events,
    GraphExploreConfig(
        congress=118,
        chamber=GraphChamberFilter.HOUSE,
        policy_area="Health",
        min_weight=2,
        max_edges=50,
    ),
)
print("House Health edges with weight >= 2:", len(filtered.edges))
draw_graph_slice(filtered, title="House Health, min weight 2")

cross_party = build_graph_slice(
    events,
    GraphExploreConfig(congress=118, cross_party_only=True, min_weight=1, max_edges=50),
)
print("Cross-party edges:", len(cross_party.edges))
draw_graph_slice(cross_party, title="Cross-party relationships only")

## Drill-down: member summary, edge evidence, ego network

In [ ]:
member_id = "S001"
summary = build_member_summary(events, member_id, BASE_CONFIG)
print("Member:", summary.label)
print("Sponsored bills:", summary.sponsored_bill_count)
print("Cosponsored bills:", summary.cosponsored_bill_count)
print("Top outgoing:", [(edge.member_id, edge.weight) for edge in summary.top_outgoing])

if graph.edges:
    edge = graph.edges[0]
    evidence = get_edge_evidence(events, edge.source, edge.target, BASE_CONFIG)
    print(f"\nEdge evidence {edge.source} → {edge.target}: weight={evidence.weight}")
    display(pd.DataFrame([bill.model_dump(mode="json") for bill in evidence.bills]))

ego = build_ego_graph(
    events,
    member_id,
    EgoNetworkConfig(congress=118, direction="outgoing", max_neighbors=10, max_edges=50),
)
draw_graph_slice(ego, title=f"Ego network: {member_id} outgoing")

## Ranked relationships and matrix view

In [ ]:
ranked = rank_relationships(events, BASE_CONFIG, top_n=10)
display(pd.DataFrame([row.model_dump(mode="json") for row in ranked]))

matrix = build_matrix(events, BASE_CONFIG, sort="party")
print(f"Matrix members: {len(matrix.rows)} | non-zero cells: {len(matrix.cells)}")

## Congress roster graph — workspace offline replay

Reload a **roster graph dataset** from `.congressgov/datasets/graphs/` and export through the **artifact lane** — no API key required when the record store already exists.

See [STORAGE.md](../docs/STORAGE.md) and [REQUEST_STORE.md](../docs/REQUEST_STORE.md).

In [ ]:
from congressgov.services.core.workspace import Workspace
from congressgov.services.export.graph import (
    GraphExploreConfig,
    export_graph_slice_to_workspace,
)

WORKSPACE = Workspace.open(".congressgov")
GRAPH_CONGRESS = 118

store = WORKSPACE.open_graph(GRAPH_CONGRESS)
if store.bill_count == 0:
    print(
        "No roster graph dataset yet. Run examples/congress_roster_graph_live.py "
        "once with CONGRESS_API_KEY, then re-run this cell offline."
    )
else:
    offline_graph = store.build_slice(
        GraphExploreConfig(
            congress=GRAPH_CONGRESS,
            min_weight=1,
            max_edges=500,
            include_all_members=True,
        )
    )
    artifact_paths = export_graph_slice_to_workspace(
        offline_graph,
        WORKSPACE,
        prefix=f"congress_roster_graph/offline_{GRAPH_CONGRESS}",
        title=f"{GRAPH_CONGRESS}th Congress roster graph (offline replay)",
    )
    print(
        f"Offline graph: {len(offline_graph.nodes)} nodes, "
        f"{len(offline_graph.edges)} edges from {store.bill_count} bills"
    )
    print("Artifacts:", artifact_paths)
    draw_graph_slice(offline_graph, title="Offline workspace graph replay")

## Export JSON for Sigma.js / downstream apps

In [ ]:
sigma_path = OUTPUT_DIR / "sample_graph_sigma.json"
api_path = OUTPUT_DIR / "sample_graph_api.json"

export_graph_slice_json(graph, sigma_path, format="sigma")
api_path.write_text(
    json.dumps(graph_slice_to_api_payload(graph), indent=2),
    encoding="utf-8",
)

print("Wrote:", sigma_path)
print("Wrote:", api_path)
display(JSON(graph_slice_to_sigma_payload(graph)))

## Live Congress.gov data

Two standalone scripts (load `CONGRESS_API_KEY` from repo `.env`):

**Ephemeral network graph** (bill batch only):

```bash
poetry run python examples/network_graph_live.py --max-bills 25
```

**Congress roster graph** (full member roster + incremental bills):

```bash
poetry run python examples/congress_roster_graph_live.py --max-bills 25
```

The cell below loads the same `.env` file and validates that the key is not the placeholder.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / ".env").exists() and (REPO_ROOT.parent / ".env").exists():
    REPO_ROOT = REPO_ROOT.parent

ENV_PATH = REPO_ROOT / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(
        f"No .env at {ENV_PATH}. Copy .env.example to .env and set CONGRESS_API_KEY."
    )

load_dotenv(ENV_PATH)
PLACEHOLDER_KEYS = {"", "your-api-key-here", "your_api_key_here"}
api_key = os.getenv("CONGRESS_API_KEY", "").strip()
if api_key.lower() in PLACEHOLDER_KEYS:
    raise ValueError(
        f"CONGRESS_API_KEY in {ENV_PATH} is still the placeholder. "
        "Paste your real key from https://api.congress.gov/sign-up/"
    )

from congressgov import Bill, Member, get_attached_store, get_client_from_env, print_store_stats
from congressgov.services.export.graph import GraphIngestionConfig, ingest_bill_events

client = get_client_from_env(load_dotenv=False)
bill_service = Bill(client=client)
member_service = Member(client=client)
member_resolver = create_member_label_resolver(member_service, congress=118, client=client)

live_events = ingest_bill_events(
    bill_service,
    GraphIngestionConfig(congress=118, bill_type="hr", batch_size=10),
    fetch_cosponsors=True,
)
print(f"Fetched {len(live_events)} sponsorship events from live bills")

store = get_attached_store(client)
if store is not None:
    print_store_stats(store)

live_graph = build_graph_slice(
    live_events,
    GraphExploreConfig(congress=118, bill_type="hr", min_weight=1, max_edges=500),
    member_resolver=member_resolver,
)
print(f"Live graph: {len(live_graph.nodes)} nodes, {len(live_graph.edges)} edges")
draw_graph_slice(live_graph, title="Live HR bills (Congress.gov API)")

## Advanced rendering

Use community detection, party-shell layouts, matrix heatmaps, analysis exports, and a standalone **Sigma.js + Graphology** HTML explorer.

In [ ]:
advanced_graph = build_graph_slice(
    events,
    GraphExploreConfig(
        congress=118,
        layout_algorithm=LayoutAlgorithm.SHELL_PARTY,
        node_color_mode=NodeColorMode.COMMUNITY,
        compute_communities=True,
        compute_advanced_metrics=True,
        min_weight=1,
        max_edges=50,
    ),
)

visualizer = GraphVisualizer(
    GraphRenderConfig(
        node_color_mode=NodeColorMode.COMMUNITY,
        label_min_centrality=0.2,
        title="Community-colored shell layout",
    )
)
display(visualizer.draw_graph(advanced_graph))

matrix = build_matrix(events, GraphExploreConfig(congress=118), sort="party")
display(visualizer.draw_matrix_heatmap(matrix))

interactive_path = OUTPUT_DIR / "network_explorer.html"
export_interactive_html(advanced_graph, interactive_path)
export_graph_slice_csv(advanced_graph, OUTPUT_DIR / "csv")
export_graph_slice_graphml(advanced_graph, OUTPUT_DIR / "graph.graphml")
print(f"Open interactive explorer: {interactive_path.resolve()}")

## Large & dense graph stress test

Synthetic data lets us probe truncation, density tiers, matrix fallback, and Sigma payload size without API calls.

In [ ]:
from congressgov.services.export.graph import (
    GraphViewMode,
    assess_graph_slice,
    generate_dense_sponsorship_events,
    profile_graph_rendering,
    recommend_explore_config,
)

dense_events = generate_dense_sponsorship_events(
    member_count=80,
    bills_per_sponsor=12,
    cosponsors_per_bill=30,
    seed=7,
)
print(f"Raw synthetic events: {len(dense_events):,}")

loose_config = GraphExploreConfig(congress=118, min_weight=1, max_edges=20000)
dense_graph_loose = build_graph_slice(dense_events, loose_config)
assessment = assess_graph_slice(dense_graph_loose)
display(assessment.model_dump(mode="json"))

safer_config = recommend_explore_config(loose_config, assessment)
dense_graph = build_graph_slice(dense_events, safer_config)
safer_assessment = assess_graph_slice(dense_graph)
print(
    f"Safer slice: {safer_assessment.node_count} nodes, "
    f"{safer_assessment.edge_count} edges, tier={safer_assessment.tier}, "
    f"recommended_view={safer_assessment.recommended_view}"
)

profile = profile_graph_rendering(dense_graph)
display(profile)

if safer_assessment.recommended_view == GraphViewMode.MATRIX:
    dense_matrix = build_matrix(dense_events, safer_config, sort="party")
    display(GraphVisualizer(GraphRenderConfig(title="Dense matrix fallback")).draw_matrix_heatmap(dense_matrix))
else:
    display(GraphVisualizer(GraphRenderConfig(title="Dense overview")).draw_graph(dense_graph))

dense_html = OUTPUT_DIR / "dense_network_explorer.html"
export_interactive_html(dense_graph, dense_html)
print(f"Dense Sigma explorer: {dense_html.resolve()}")

## Interpretation notes

- Cosponsorship is support for a bill, not proof of ideological alignment.
- High edge weights may reflect popular or symbolic legislation.
- Cross-party ties are useful signals, not complete bipartisanship measures.
- Withdrawn cosponsorships are excluded from active weights by default.
- Graphs may be truncated for readability — check `graph.limits`.

See [NETWORK_GRAPH.md](../docs/NETWORK_GRAPH.md) for the full design reference.